<a href="https://colab.research.google.com/github/Danieladunaev/Danieladunaev/blob/main/Representative_Sample_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Representative Sample Agent

**Purpose:** assess whether a 70% development sample is representative of the full synthetic portfolio and whether repeated random 70/30 splits produce stable data distributions and downstream model outputs.

This notebook is designed for Google Colab and Model Risk Management review. It uses **synthetic data only**, fixes random seeds for reproducibility, separates statistical significance from practical materiality, and exports audit-friendly result tables.

> Scope note: representativeness here means similarity to the synthetic source population under the variables and process explicitly generated below. It does not prove representativeness of any real Bank of Israel population or future production data.

## 1. Methodology and decision rules

The agent performs five linked checks:

1. Generate a clean synthetic credit-risk portfolio with a binary `default` target.
2. Create one baseline stratified 70/30 train-test split.
3. Compare train, test, and full-population distributions using standardized mean differences (SMD), Kolmogorov-Smirnov statistics, categorical proportion gaps, and population stability index (PSI).
4. Repeat the entire split-and-model workflow across alternative random seeds to measure sampling sensitivity.
5. Bootstrap the baseline training sample, refit the pipeline, and evaluate each refit on the same locked test set to isolate training-sample uncertainty.

**Pre-specified practical flags (illustrative, not universal):**

- absolute SMD: `< 0.10` negligible, `0.10–0.20` review, `> 0.20` material;
- PSI: `< 0.10` stable, `0.10–0.25` review, `> 0.25` material;
- categorical maximum share gap: `> 5 percentage points` review;
- target-rate gap: `> 2 percentage points` review;
- model AUC range across repeated splits: `> 0.03` review.

These thresholds are governance choices. Calibrate them to portfolio size, default prevalence, model use, and the institution's policy before production use.

In [ ]:
# Colab already contains these packages. Uncomment only if a package is missing.
# %pip install -q pandas numpy scipy scikit-learn seaborn matplotlib

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", palette="colorblind")

SEED = 42
N_ROWS = 12_000
TRAIN_SHARE = 0.70
TARGET = "default"
SPLIT_COL = "split"

SMD_REVIEW = 0.10
SMD_MATERIAL = 0.20
PSI_REVIEW = 0.10
PSI_MATERIAL = 0.25
CATEGORY_GAP_REVIEW_PP = 5.0
TARGET_GAP_REVIEW_PP = 2.0
AUC_RANGE_REVIEW = 0.03

## 2. Generate a clean synthetic portfolio

The generator creates plausible dependencies rather than independent random columns. For example, debt-to-income depends partly on income and loan size, while default probability depends on score, leverage, utilization, delinquencies, employment length, and loan purpose. This makes the downstream stability exercise meaningful without using confidential data.

In [ ]:
rng = np.random.default_rng(SEED)
n = N_ROWS

age = np.clip(rng.normal(43, 12, n).round(), 18, 79).astype(int)
annual_income = np.clip(rng.lognormal(np.log(85_000), 0.48, n), 18_000, 450_000)
employment_years = np.clip(age - 18 - rng.gamma(2.0, 2.2, n), 0, 45).round(1)
loan_amount = np.clip(rng.lognormal(np.log(70_000), 0.58, n), 5_000, 400_000)
debt_to_income = np.clip(
    0.10 + 0.34 * loan_amount / annual_income + rng.beta(2.2, 6.0, n) * 0.45,
    0.03, 0.95,
)
credit_utilization = np.clip(
    0.12 + 0.52 * debt_to_income + rng.normal(0, 0.16, n), 0.0, 1.0
)
delinquencies_2y = np.clip(
    rng.poisson(0.18 + 1.15 * credit_utilization), 0, 8
).astype(int)
credit_score = np.clip(
    770 - 145 * credit_utilization - 18 * delinquencies_2y
    + 0.45 * (age - 43) + rng.normal(0, 34, n),
    300, 850,
).round().astype(int)
region = rng.choice(["Central", "North", "South", "Jerusalem"], n,
                    p=[0.46, 0.22, 0.18, 0.14])
loan_purpose = rng.choice(["Mortgage", "Business", "Vehicle", "Personal"], n,
                          p=[0.42, 0.20, 0.18, 0.20])
underwriting_year = rng.choice(np.arange(2020, 2026), n,
                               p=[0.13, 0.15, 0.17, 0.18, 0.19, 0.18])

logit = (
    -2.65
    + 2.15 * debt_to_income
    + 1.55 * credit_utilization
    + 0.29 * delinquencies_2y
    - 0.010 * (credit_score - 650)
    - 0.018 * employment_years
    + 0.30 * (loan_purpose == "Personal")
    + 0.18 * (loan_purpose == "Business")
    + 0.10 * (region == "South")
)
probability_default = 1 / (1 + np.exp(-logit))
default = rng.binomial(1, probability_default)

df = pd.DataFrame({
    "client_id": [f"SYN-{i:06d}" for i in range(1, n + 1)],
    "age": age,
    "annual_income": annual_income.round(2),
    "employment_years": employment_years,
    "loan_amount": loan_amount.round(2),
    "debt_to_income": debt_to_income.round(4),
    "credit_utilization": credit_utilization.round(4),
    "delinquencies_2y": delinquencies_2y,
    "credit_score": credit_score,
    "region": region,
    "loan_purpose": loan_purpose,
    "underwriting_year": underwriting_year,
    TARGET: default,
})

print(f"Synthetic rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Default rate: {df[TARGET].mean():.2%}")
display(df.head())

## 3. Data-quality gates and baseline 70/30 split

Stratification keeps the target proportion approximately equal across train and test. The `split` column is written back to the dataset so each row's assignment is explicit and auditable.

In [ ]:
quality_checks = pd.DataFrame({
    "check": ["No missing cells", "Unique client IDs", "Binary target", "Finite numeric values"],
    "passed": [
        df.isna().sum().sum() == 0,
        df["client_id"].is_unique,
        set(df[TARGET].unique()) == {0, 1},
        np.isfinite(df.select_dtypes(include=np.number).to_numpy()).all(),
    ],
    "detail": [
        f"Missing cells: {df.isna().sum().sum():,}",
        f"Duplicate IDs: {df['client_id'].duplicated().sum():,}",
        f"Observed: {sorted(df[TARGET].unique())}",
        "All numeric values finite",
    ],
})
display(quality_checks)
assert quality_checks["passed"].all(), "Synthetic data-quality gate failed."

train_idx, test_idx = train_test_split(
    df.index, train_size=TRAIN_SHARE, random_state=SEED, stratify=df[TARGET]
)
df[SPLIT_COL] = "test"
df.loc[train_idx, SPLIT_COL] = "train"

split_summary = df.groupby(SPLIT_COL)[TARGET].agg(rows="size", defaults="sum", default_rate="mean")
split_summary["share"] = split_summary["rows"] / len(df)
display(split_summary.style.format({"default_rate": "{:.2%}", "share": "{:.2%}"}))
assert len(train_idx) == int(TRAIN_SHARE * len(df))

## 4. EDA: full portfolio and split comparisons

Visual overlap is useful for diagnosis, but the formal tables in the next section drive the decision. With a large sample, tiny differences can be statistically significant; practical effect sizes are therefore emphasized.

In [ ]:
numeric_features = [
    "age", "annual_income", "employment_years", "loan_amount",
    "debt_to_income", "credit_utilization", "delinquencies_2y", "credit_score",
]
categorical_features = ["region", "loan_purpose", "underwriting_year"]

display(df[numeric_features + [TARGET]].describe().T)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, feature in zip(axes.flat, ["credit_score", "debt_to_income", "annual_income", "loan_amount"]):
    sns.kdeplot(data=df, x=feature, hue=SPLIT_COL, common_norm=False, ax=ax)
    ax.set_title(f"{feature}: train vs test")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="region", hue=SPLIT_COL, stat="proportion", common_norm=False, ax=axes[0])
sns.countplot(data=df, x="loan_purpose", hue=SPLIT_COL, stat="proportion", common_norm=False, ax=axes[1])
axes[0].set_title("Region mix by split")
axes[1].set_title("Loan-purpose mix by split")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 5. Representativeness tests for the baseline split

- **SMD** measures mean separation in pooled-standard-deviation units and is less sample-size-sensitive than a p-value.
- **KS** tests whether two continuous empirical distributions differ anywhere, but its p-value can flag immaterial differences in large datasets.
- **Maximum category share gap** is the largest absolute train-test difference among a feature's levels.
- **PSI** summarizes train-test distribution movement across fixed bins. It is diagnostic and depends on binning; it is not a formal hypothesis test.

In [ ]:
def standardized_mean_difference(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled_sd = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return 0.0 if pooled_sd == 0 else (np.mean(a) - np.mean(b)) / pooled_sd

def psi(reference, comparison, bins=10, epsilon=1e-6):
    reference, comparison = np.asarray(reference), np.asarray(comparison)
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct = np.histogram(reference, bins=edges)[0] / len(reference)
    cmp_pct = np.histogram(comparison, bins=edges)[0] / len(comparison)
    ref_pct = np.clip(ref_pct, epsilon, None)
    cmp_pct = np.clip(cmp_pct, epsilon, None)
    return np.sum((cmp_pct - ref_pct) * np.log(cmp_pct / ref_pct))

train = df.loc[df[SPLIT_COL] == "train"]
test = df.loc[df[SPLIT_COL] == "test"]

numeric_balance = []
for feature in numeric_features:
    ks = stats.ks_2samp(train[feature], test[feature])
    smd = standardized_mean_difference(train[feature], test[feature])
    psi_value = psi(train[feature], test[feature])
    numeric_balance.append({
        "feature": feature,
        "train_mean": train[feature].mean(),
        "test_mean": test[feature].mean(),
        "smd": smd,
        "abs_smd": abs(smd),
        "ks_statistic": ks.statistic,
        "ks_p_value": ks.pvalue,
        "psi": psi_value,
        "practical_flag": abs(smd) >= SMD_REVIEW or psi_value >= PSI_REVIEW,
    })
numeric_balance = pd.DataFrame(numeric_balance).sort_values("abs_smd", ascending=False)

categorical_balance = []
for feature in categorical_features:
    shares = pd.crosstab(df[feature], df[SPLIT_COL], normalize="columns").fillna(0)
    gap_pp = 100 * (shares["train"] - shares["test"]).abs()
    contingency = pd.crosstab(df[feature], df[SPLIT_COL])
    chi2, p_value, _, _ = stats.chi2_contingency(contingency)
    categorical_balance.append({
        "feature": feature,
        "max_share_gap_pp": gap_pp.max(),
        "chi_square_p_value": p_value,
        "practical_flag": gap_pp.max() >= CATEGORY_GAP_REVIEW_PP,
    })
categorical_balance = pd.DataFrame(categorical_balance).sort_values("max_share_gap_pp", ascending=False)

target_gap_pp = 100 * abs(train[TARGET].mean() - test[TARGET].mean())
display(numeric_balance.style.format({"train_mean": "{:.3f}", "test_mean": "{:.3f}", "smd": "{:.3f}", "abs_smd": "{:.3f}", "ks_statistic": "{:.3f}", "ks_p_value": "{:.3g}", "psi": "{:.3f}"}))
display(categorical_balance.style.format({"max_share_gap_pp": "{:.2f}", "chi_square_p_value": "{:.3g}"}))
print(f"Absolute default-rate gap: {target_gap_pp:.3f} percentage points")

## 6. Define the downstream pipeline

The same preprocessing and logistic-regression pipeline is used in every repeated split and bootstrap replicate. AUC measures ranking performance; Brier score measures probability error (lower is better). This is a demonstration pipeline, not a production credit model.

In [ ]:
feature_columns = numeric_features + categorical_features

def make_pipeline():
    preprocessor = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1_000, random_state=SEED)),
    ])

baseline_model = make_pipeline()
baseline_model.fit(train[feature_columns], train[TARGET])
baseline_probability = baseline_model.predict_proba(test[feature_columns])[:, 1]
baseline_metrics = pd.Series({
    "test_auc": roc_auc_score(test[TARGET], baseline_probability),
    "test_brier": brier_score_loss(test[TARGET], baseline_probability),
    "train_default_rate": train[TARGET].mean(),
    "test_default_rate": test[TARGET].mean(),
}, name="baseline")
display(baseline_metrics.to_frame())

## 7. Alternative 70% random splits

Each iteration draws a new stratified 70% training sample, evaluates balance, fits the same pipeline, and scores its corresponding 30% test sample. This measures the combined sensitivity of sample composition, model fitting, and test-set composition.

A stable process should show narrow metric ranges and few practical balance flags. If instability appears, investigate class rarity, influential subgroups, temporal structure, leakage, or insufficient sample size rather than simply trying seeds until a favorable split appears.

In [ ]:
N_SPLITS = 100  # Increase for a stricter production review if runtime permits.
repeated_results = []

for split_seed in range(1, N_SPLITS + 1):
    tr_idx, te_idx = train_test_split(
        df.index, train_size=TRAIN_SHARE, random_state=split_seed, stratify=df[TARGET]
    )
    tr, te = df.loc[tr_idx], df.loc[te_idx]
    max_abs_smd = max(abs(standardized_mean_difference(tr[c], te[c])) for c in numeric_features)
    max_psi = max(psi(tr[c], te[c]) for c in numeric_features)
    model = make_pipeline()
    model.fit(tr[feature_columns], tr[TARGET])
    probability = model.predict_proba(te[feature_columns])[:, 1]
    repeated_results.append({
        "seed": split_seed,
        "max_abs_smd": max_abs_smd,
        "max_psi": max_psi,
        "default_rate_gap_pp": 100 * abs(tr[TARGET].mean() - te[TARGET].mean()),
        "test_auc": roc_auc_score(te[TARGET], probability),
        "test_brier": brier_score_loss(te[TARGET], probability),
    })

repeated_results = pd.DataFrame(repeated_results)
repeated_summary = repeated_results.drop(columns="seed").agg(["mean", "std", "min", "max"]).T
repeated_summary["range"] = repeated_summary["max"] - repeated_summary["min"]
display(repeated_summary.style.format("{:.4f}"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(repeated_results["max_abs_smd"], kde=True, ax=axes[0])
axes[0].axvline(SMD_REVIEW, color="red", linestyle="--", label="review threshold")
axes[0].set_title("Worst numeric SMD by split")
sns.histplot(repeated_results["test_auc"], kde=True, ax=axes[1])
axes[1].set_title("Test AUC across splits")
sns.scatterplot(data=repeated_results, x="max_abs_smd", y="test_auc", ax=axes[2])
axes[2].set_title("Balance vs model output")
axes[0].legend()
plt.tight_layout()
plt.show()

## 8. Bootstrap concept and application

**Bootstrapping** repeatedly samples rows *with replacement* from an observed sample, keeping the same sample size. The empirical distribution of a statistic across bootstrap replicates approximates its sampling uncertainty.

Here we bootstrap the fixed baseline training set, refit the pipeline, and always evaluate on the same locked baseline test set. This isolates how much the fitted model changes when the development sample is perturbed. We also bootstrap the train-vs-full-population maximum SMD as a direct representativeness diagnostic.

This complements—but does not replace—the repeated-split exercise:

- repeated splits vary both development and evaluation membership;
- the bootstrap varies training membership while keeping evaluation fixed;
- neither method repairs selection bias or distribution shift not present in the synthetic generator;
- for time-dependent data, use out-of-time validation or block bootstrap instead of ordinary row bootstrap.

In [ ]:
N_BOOTSTRAPS = 200
boot_rng = np.random.default_rng(SEED + 10_000)
bootstrap_results = []

for replicate in range(1, N_BOOTSTRAPS + 1):
    boot_positions = boot_rng.integers(0, len(train), size=len(train))
    boot = train.iloc[boot_positions]
    model = make_pipeline()
    model.fit(boot[feature_columns], boot[TARGET])
    probability = model.predict_proba(test[feature_columns])[:, 1]
    max_population_smd = max(
        abs(standardized_mean_difference(boot[c], df[c])) for c in numeric_features
    )
    bootstrap_results.append({
        "replicate": replicate,
        "test_auc": roc_auc_score(test[TARGET], probability),
        "test_brier": brier_score_loss(test[TARGET], probability),
        "bootstrap_default_rate": boot[TARGET].mean(),
        "max_abs_smd_vs_population": max_population_smd,
    })

bootstrap_results = pd.DataFrame(bootstrap_results)

def interval_table(data, columns):
    rows = []
    for column in columns:
        rows.append({
            "metric": column,
            "estimate": data[column].mean(),
            "bootstrap_std_error": data[column].std(ddof=1),
            "ci_2.5%": data[column].quantile(0.025),
            "ci_97.5%": data[column].quantile(0.975),
            "interval_width": data[column].quantile(0.975) - data[column].quantile(0.025),
        })
    return pd.DataFrame(rows)

bootstrap_intervals = interval_table(
    bootstrap_results,
    ["test_auc", "test_brier", "bootstrap_default_rate", "max_abs_smd_vs_population"],
)
display(bootstrap_intervals.style.format({c: "{:.4f}" for c in bootstrap_intervals.columns if c != "metric"}))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(
    axes,
    ["test_auc", "test_brier", "max_abs_smd_vs_population"],
    ["Bootstrap AUC", "Bootstrap Brier score", "Bootstrap max SMD vs population"],
):
    sns.histplot(bootstrap_results[metric], kde=True, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 9. Automated MRM-oriented conclusion

The verdict combines practical thresholds rather than relying on a single p-value. A `REVIEW` result is not automatic rejection; it identifies the dimension that needs investigation and documentation.

In [ ]:
auc_range = repeated_results["test_auc"].max() - repeated_results["test_auc"].min()
baseline_max_smd = numeric_balance["abs_smd"].max()
baseline_max_psi = numeric_balance["psi"].max()
baseline_max_category_gap = categorical_balance["max_share_gap_pp"].max()

decision_table = pd.DataFrame([
    ["Baseline maximum absolute SMD", baseline_max_smd, SMD_REVIEW, baseline_max_smd < SMD_REVIEW],
    ["Baseline maximum PSI", baseline_max_psi, PSI_REVIEW, baseline_max_psi < PSI_REVIEW],
    ["Baseline maximum category gap (pp)", baseline_max_category_gap, CATEGORY_GAP_REVIEW_PP, baseline_max_category_gap < CATEGORY_GAP_REVIEW_PP],
    ["Baseline default-rate gap (pp)", target_gap_pp, TARGET_GAP_REVIEW_PP, target_gap_pp < TARGET_GAP_REVIEW_PP],
    ["Repeated-split AUC range", auc_range, AUC_RANGE_REVIEW, auc_range < AUC_RANGE_REVIEW],
], columns=["check", "observed", "review_threshold", "passed"])
overall_decision = "PASS" if decision_table["passed"].all() else "REVIEW"
display(decision_table.style.format({"observed": "{:.4f}", "review_threshold": "{:.4f}"}))
print(f"Overall illustrative decision: {overall_decision}")
print("Interpretation: PASS supports stability under this synthetic design; it does not establish external representativeness.")

## 10. Export audit artifacts

The synthetic row-level file and all review tables are saved under `representative_sample_outputs/`. In Colab, download the files from the Files pane or commit selected artifacts to GitHub according to your institution's repository policy.

In [ ]:
output_dir = Path("representative_sample_outputs")
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / "synthetic_representative_sample.csv", index=False)
quality_checks.to_csv(output_dir / "quality_checks.csv", index=False)
split_summary.to_csv(output_dir / "baseline_split_summary.csv")
numeric_balance.to_csv(output_dir / "baseline_numeric_balance.csv", index=False)
categorical_balance.to_csv(output_dir / "baseline_categorical_balance.csv", index=False)
repeated_results.to_csv(output_dir / "repeated_split_results.csv", index=False)
bootstrap_results.to_csv(output_dir / "bootstrap_results.csv", index=False)
bootstrap_intervals.to_csv(output_dir / "bootstrap_intervals.csv", index=False)
decision_table.to_csv(output_dir / "decision_table.csv", index=False)

print(f"Saved {len(list(output_dir.glob('*.csv')))} CSV files to {output_dir.resolve()}")

## 11. Exact Colab ↔ GitHub workflow

### Recommended first-time setup (Colab interface)

1. On GitHub, create a repository (for example, `mrm-representative-sample`). Do **not** include confidential or real customer data.
2. In Colab, choose **File → Save a copy in GitHub**.
3. Authorize Colab when prompted, select the repository and branch, set the path to `notebooks/representative_sample_agent.ipynb`, enter a commit message, and click **OK**.
4. For later work, open `https://colab.research.google.com`, choose **GitHub**, select the repository and notebook, run/edit it, then repeat **File → Save a copy in GitHub**.
5. Confirm on GitHub that the commit and notebook render correctly. Review the diff before merging to a protected branch.

### Reproducible Git workflow inside Colab

Use this when you need normal Git branches and commits. Create a **fine-grained GitHub personal access token** with access only to the target repository and the minimum Contents permission required. Never paste a token into a notebook cell, output, remote URL, or committed file. Colab runtimes are temporary, so repeat the clone/authentication setup after a runtime reset.

1. Set `REPO_URL`, `BRANCH`, and Git identity in the next cell.
2. Run the cell and enter the token only in the hidden prompt.
3. Copy or save the notebook into the cloned repository's `notebooks/` folder.
4. Review `git status` and `git diff`, then commit and push.
5. Revoke the token if it was exposed. Follow Bank of Israel policy for approved GitHub organization, visibility, and artifact retention.

> If your organization uses SSH, GitHub Apps, SSO, or a managed credential helper, use that approved method instead of a personal token.

In [ ]:
# OPTIONAL COLAB-ONLY CELL. Set RUN_GIT_SYNC=True after reviewing every value.
RUN_GIT_SYNC = False

if RUN_GIT_SYNC:
    import getpass
    import os
    import shutil
    import subprocess
    import tempfile

    REPO_URL = "https://github.com/YOUR-ORG/mrm-representative-sample.git"
    REPO_DIR = "/content/mrm-representative-sample"
    BRANCH = "feature/representative-sample-agent"
    GIT_NAME = "YOUR NAME"
    GIT_EMAIL = "YOUR_EMAIL@example.com"
    NOTEBOOK_SOURCE = "/content/Representative_Sample_Agent.ipynb"

    token = getpass.getpass("GitHub fine-grained token (input is hidden): ")
    askpass_file = tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False)
    askpass_file.write(
        "import os, sys\n"
        "print('x-access-token' if 'Username' in sys.argv[1] else os.environ['COLAB_GITHUB_TOKEN'])\n"
    )
    askpass_file.close()
    auth_env = os.environ.copy()
    auth_env["GIT_ASKPASS"] = askpass_file.name
    auth_env["GIT_TERMINAL_PROMPT"] = "0"
    auth_env["COLAB_GITHUB_TOKEN"] = token
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], env=auth_env, check=True)
    subprocess.run(["git", "config", "user.name", GIT_NAME], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "user.email", GIT_EMAIL], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "switch", "-c", BRANCH], cwd=REPO_DIR, check=True)

    notebooks_dir = Path(REPO_DIR) / "notebooks"
    notebooks_dir.mkdir(exist_ok=True)
    shutil.copy2(NOTEBOOK_SOURCE, notebooks_dir / "representative_sample_agent.ipynb")
    shutil.copytree(output_dir, Path(REPO_DIR) / output_dir.name, dirs_exist_ok=True)

    subprocess.run(["git", "status", "--short"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "add", "notebooks/representative_sample_agent.ipynb"], cwd=REPO_DIR, check=True)
    # Add generated CSVs only if repository policy explicitly permits them:
    # subprocess.run(["git", "add", "representative_sample_outputs"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "commit", "-m", "Add representative sample stability agent"], cwd=REPO_DIR, check=True)

    subprocess.run(["git", "push", "-u", "origin", BRANCH], cwd=REPO_DIR, env=auth_env, check=True)
    os.unlink(askpass_file.name)
    auth_env.pop("COLAB_GITHUB_TOKEN", None)
    token = None
    print("Push complete. Open GitHub, inspect the diff, and create a pull request.")
else:
    print("Git sync is disabled. Review the instructions, fill placeholders, then set RUN_GIT_SYNC=True in Colab.")

## 12. Limitations and governance notes

- Synthetic data validates code behavior, not real-world portfolio representativeness.
- Random splitting assumes rows are exchangeable. Use customer-grouped splits for repeated customers and out-of-time splits where time drift matters.
- Ordinary bootstrap assumes independent observations; clustered or time-dependent portfolios require cluster/block bootstrap.
- Multiple diagnostics are intentionally reported. Avoid treating p-values as pass/fail evidence without effect size and business materiality.
- Record the generator version, seeds, thresholds, package versions, and any threshold overrides in the model-development documentation.
- Keep real data, credentials, notebook outputs, and logs out of Git unless explicitly approved.